# Spark Data Types and Casting

Raw files frequently represent every value as text. A reliable Spark DataFrame gives each column an appropriate type before it is used for calculations, filtering, or storage.

---
## Learning objectives

By the end of this notebook, you will be able to:

- inspect a DataFrame schema and identify common Spark data types;
- convert text values to numeric, decimal, boolean, date, and timestamp types;
- handle invalid values that become `NULL` during a safe conversion; and
- prepare a raw, string-based DataFrame for analysis.

---
## Inspect the raw schema

This sales extract has arrived as text. `printSchema()` tells us how Spark currently understands every column.

In [2]:
%run "./0 - Create spark session.ipynb"

In [3]:
from pyspark.sql import functions as F

sales_raw = spark.createDataFrame(
    [
        ("1001", "24.50", "2", "true", "2024-01-15", "2024-01-15 08:30:00"),
        ("1002", "19.99", "1", "false", "2024-01-16", "2024-01-16 13:45:00"),
        ("1003", "not available", "three", "unknown", "2024-01-17", "2024-01-17 17:00:00"),
    ],
    ["order_id_text", "unit_price_text", "quantity_text", "priority_text", "order_date_text", "created_at_text"],
)
sales_raw.show()
sales_raw.printSchema()

+-------------+---------------+-------------+-------------+---------------+-------------------+
|order_id_text|unit_price_text|quantity_text|priority_text|order_date_text|    created_at_text|
+-------------+---------------+-------------+-------------+---------------+-------------------+
|         1001|          24.50|            2|         true|     2024-01-15|2024-01-15 08:30:00|
|         1002|          19.99|            1|        false|     2024-01-16|2024-01-16 13:45:00|
|         1003|  not available|        three|      unknown|     2024-01-17|2024-01-17 17:00:00|
+-------------+---------------+-------------+-------------+---------------+-------------------+

root
 |-- order_id_text: string (nullable = true)
 |-- unit_price_text: string (nullable = true)
 |-- quantity_text: string (nullable = true)
 |-- priority_text: string (nullable = true)
 |-- order_date_text: string (nullable = true)
 |-- created_at_text: string (nullable = true)



---
## Cast valid values explicitly

`cast` changes a column's type. Use an integer for whole-number identifiers and a boolean for true/false flags. For currency, use a fixed-precision `decimal` instead of a floating-point type so cents are represented as expected.

In [6]:
valid_sales = sales_raw.filter(F.col("order_id_text") != "1003").select(
    F.col("order_id_text").cast("int").alias("order_id"),
    F.col("unit_price_text").cast("decimal(10,2)").alias("unit_price"),
    F.col("quantity_text").cast("int").alias("quantity"),
    F.col("priority_text").cast("boolean").alias("is_priority"),
    F.to_date("order_date_text", "yyyy-MM-dd").alias("order_date"),
    F.to_timestamp("created_at_text", "yyyy-MM-dd HH:mm:ss").alias("created_at"),
)
valid_sales.show()
valid_sales.printSchema()

+--------+----------+--------+-----------+----------+-------------------+
|order_id|unit_price|quantity|is_priority|order_date|         created_at|
+--------+----------+--------+-----------+----------+-------------------+
|    1001|     24.50|       2|       true|2024-01-15|2024-01-15 08:30:00|
|    1002|     19.99|       1|      false|2024-01-16|2024-01-16 13:45:00|
+--------+----------+--------+-----------+----------+-------------------+

root
 |-- order_id: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- is_priority: boolean (nullable = true)
 |-- order_date: date (nullable = true)
 |-- created_at: timestamp (nullable = true)



---
## Safely convert messy data

Real extracts can contain invalid values. `try_cast` is a safe SQL conversion: instead of failing when a value cannot be converted, it returns `NULL`. This lets us find and handle bad data deliberately.

In [7]:
sales_with_safe_casts = sales_raw.select(
    F.col("order_id_text").cast("int").alias("order_id"),
    F.expr("try_cast(unit_price_text AS DECIMAL(10,2))").alias("unit_price"),
    F.expr("try_cast(quantity_text AS INT)").alias("quantity"),
    F.expr("try_cast(priority_text AS BOOLEAN)").alias("is_priority"),
    F.to_date("order_date_text", "yyyy-MM-dd").alias("order_date"),
    F.to_timestamp("created_at_text", "yyyy-MM-dd HH:mm:ss").alias("created_at"),
)
sales_with_safe_casts.show()
sales_with_safe_casts.printSchema()

+--------+----------+--------+-----------+----------+-------------------+
|order_id|unit_price|quantity|is_priority|order_date|         created_at|
+--------+----------+--------+-----------+----------+-------------------+
|    1001|     24.50|       2|       true|2024-01-15|2024-01-15 08:30:00|
|    1002|     19.99|       1|      false|2024-01-16|2024-01-16 13:45:00|
|    1003|      NULL|    NULL|       NULL|2024-01-17|2024-01-17 17:00:00|
+--------+----------+--------+-----------+----------+-------------------+

root
 |-- order_id: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- is_priority: boolean (nullable = true)
 |-- order_date: date (nullable = true)
 |-- created_at: timestamp (nullable = true)



---
## Find and handle conversion failures

A `NULL` after a safe conversion is a data-quality signal. Keep raw values beside converted values, then mark rows that need review.

In [8]:
quality_check = sales_raw.select(
    "order_id_text",
    "unit_price_text",
    F.expr("try_cast(unit_price_text AS DECIMAL(10,2))").alias("unit_price"),
    "quantity_text",
    F.expr("try_cast(quantity_text AS INT)").alias("quantity"),
).withColumn(
    "needs_review",
    F.col("unit_price").isNull() | F.col("quantity").isNull(),
)
quality_check.show()
quality_check.filter("needs_review").show()

+-------------+---------------+----------+-------------+--------+------------+
|order_id_text|unit_price_text|unit_price|quantity_text|quantity|needs_review|
+-------------+---------------+----------+-------------+--------+------------+
|         1001|          24.50|     24.50|            2|       2|       false|
|         1002|          19.99|     19.99|            1|       1|       false|
|         1003|  not available|      NULL|        three|    NULL|        true|
+-------------+---------------+----------+-------------+--------+------------+

+-------------+---------------+----------+-------------+--------+------------+
|order_id_text|unit_price_text|unit_price|quantity_text|quantity|needs_review|
+-------------+---------------+----------+-------------+--------+------------+
|         1003|  not available|      NULL|        three|    NULL|        true|
+-------------+---------------+----------+-------------+--------+------------+



---
## Your turn: clean a product extract

Create `products_typed` from `products_raw`. Convert `product_id_text` to an integer, `price_text` to `decimal(10,2)`, `in_stock_text` to boolean, and `last_updated_text` to a timestamp. Add a boolean `needs_review` that is true when the price could not be converted. Preview the result and schema.

In [ ]:
products_raw = spark.createDataFrame(
    [
        ("501", "12.50", "true", "2024-02-01 10:00:00"),
        ("502", "price pending", "false", "2024-02-01 11:30:00"),
    ],
    ["product_id_text", "price_text", "in_stock_text", "last_updated_text"],
)

# Write your solution here.

---
## Key takeaway

Inspect the schema after reading data, convert every field deliberately, and treat failed conversions as data-quality information. Arrays, maps, and structs are useful Spark types too, but belong in a later advanced lesson.